In [0]:
#Read silver table
silver_df = spark.table("workspace.default.silver_hhs_healthcare_breaches")

print("Final Silver Records:", silver_df.count())
display(silver_df)

Final Silver Records: 723


covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
"CardioFit Medical Group, Inc.",CA,Healthcare Provider,7243,2026-04-09,Unauthorized Access/Disclosure,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,Medium,Available
Ideal Home Care,OK,Healthcare Provider,1331,2026-03-13,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,3,High,Available
Cedar Valley Services,MN,Healthcare Provider,501,2026-02-13,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,2,High,Available
Triad Radiology Associates,NC,Healthcare Provider,11011,2026-02-06,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,2,High,Available
Andover Eye Associates,MA,Healthcare Provider,1638,2025-12-31,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,12,High,Available
"Vida Y Salud-Health Systems, Inc.",TX,Healthcare Provider,35236,2025-12-08,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,12,High,Available
"Fieldtex Products, Inc.",NY,Business Associate,238615,2025-11-20,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,11,High,Available
Denton MHMR Center,TX,Healthcare Provider,108967,2025-11-05,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,11,High,Available
Beverly Hills Oncology Medical Group,CA,Healthcare Provider,57655,2025-10-31,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,10,High,Available


In [0]:
#risk level summary
gold_risk_summary = spark.sql("""
SELECT
    risk_level,
    COUNT(*) AS total_breaches,
    SUM(individuals_affected) AS total_individuals_affected,
    ROUND(AVG(individuals_affected), 2) AS avg_individuals_affected
FROM workspace.default.silver_hhs_healthcare_breaches
GROUP BY risk_level
ORDER BY total_breaches DESC
""")

display(gold_risk_summary)

risk_level,total_breaches,total_individuals_affected,avg_individuals_affected
High,631,288389206,457035.19
Medium,85,527511,6206.01
Low,7,25865,3695.0


In [0]:
gold_risk_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_risk_summary"
)

In [0]:
#breach type summary
gold_breach_type_summary = spark.sql("""
SELECT
    type_of_breach,
    COUNT(*) AS total_breaches,
    SUM(individuals_affected) AS total_individuals_affected,
    ROUND(AVG(individuals_affected), 2) AS avg_individuals_affected
FROM workspace.default.silver_hhs_healthcare_breaches
GROUP BY type_of_breach
ORDER BY total_breaches DESC
""")

display(gold_breach_type_summary)

type_of_breach,total_breaches,total_individuals_affected,avg_individuals_affected
Hacking/IT Incident,630,288085241,457278.16
Unauthorized Access/Disclosure,85,796801,9374.13
Theft,6,25044,4174.0
Loss,1,821,821.0
Improper Disposal,1,34675,34675.0


In [0]:
gold_breach_type_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_breach_type_summary"
)

In [0]:
#Monthly breach 
gold_monthly_breach_trend = spark.sql("""
SELECT
    breach_year,
    breach_month,
    COUNT(*) AS total_breaches,
    SUM(individuals_affected) AS total_individuals_affected
FROM workspace.default.silver_hhs_healthcare_breaches
GROUP BY breach_year, breach_month
ORDER BY breach_year, breach_month
""")

display(gold_monthly_breach_trend)

breach_year,breach_month,total_breaches,total_individuals_affected
2023,10,1,10833
2023,11,2,2280
2023,12,1,10079
2024,2,2,30497
2024,3,2,217670
2024,4,3,1038178
2024,5,8,3368317
2024,6,8,577730
2024,7,14,199865606
2024,8,18,8855621


In [0]:
gold_monthly_breach_trend.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_monthly_breach_trend"
)

In [0]:
#State wise breach analysis
gold_state_breach_summary = spark.sql("""
SELECT
    state,
    COUNT(*) AS total_breaches,
    SUM(individuals_affected) AS total_individuals_affected,
    ROUND(AVG(individuals_affected), 2) AS avg_individuals_affected
FROM workspace.default.silver_hhs_healthcare_breaches
GROUP BY state
ORDER BY total_breaches DESC
""")

display(gold_state_breach_summary)

state,total_breaches,total_individuals_affected,avg_individuals_affected
CA,64,13227075,206673.05
TX,57,4514432,79200.56
FL,53,3307578,62407.13
NY,45,2270579,50457.31
PA,31,1453930,46900.97
IL,30,3661599,122053.3
MA,23,548205,23835.0
MI,22,1114313,50650.59
WA,20,592246,29612.3
OH,20,2147984,107399.2


In [0]:
gold_state_breach_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_state_breach_summary"
)

In [0]:
#High risk incidents

gold_high_risk_incidents = spark.sql("""
SELECT
    covered_entity_name,
    state,
    covered_entity_type,
    individuals_affected,
    breach_submission_date,
    type_of_breach,
    location_of_breached_information,
    risk_level
FROM workspace.default.silver_hhs_healthcare_breaches
WHERE risk_level = 'High'
ORDER BY individuals_affected DESC
""")

display(gold_high_risk_incidents)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,risk_level
"Change Healthcare, Inc.",MN,Business Associate,192700000,2024-07-19,Hacking/IT Incident,Network Server,High
Aflac Incorporated (“Aflac”),GA,Health Plan,13924906,2025-08-08,Hacking/IT Incident,Network Server,High
"Episource, LLC",CA,Business Associate,6725572,2025-06-06,Hacking/IT Incident,Network Server,High
Ascension Health,MO,Healthcare Provider,5466931,2024-07-03,Hacking/IT Incident,Network Server,High
"HealthEquity, Inc.",UT,Business Associate,4300000,2024-08-09,Hacking/IT Incident,Network Server,High
TriZetto Provider Solutions,MO,Business Associate,3433965,2026-02-06,Hacking/IT Incident,Network Server,High
"QualDerm Partners, LLC",TN,Healthcare Provider,3117874,2026-02-22,Hacking/IT Incident,Network Server,High
"PIH Health, Inc.",CA,Healthcare Provider,2947264,2025-01-31,Hacking/IT Incident,Network Server,High
"Acadian Ambulance Service, Inc.",LA,Healthcare Provider,2896985,2024-08-20,Hacking/IT Incident,Network Server,High
A&A Services d/b/a Sav-Rx,NE,Business Associate,2812336,2024-05-24,Hacking/IT Incident,Network Server,High


In [0]:
gold_high_risk_incidents.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_high_risk_incidents"
)